In [7]:
import xml.etree.ElementTree as ET
import os
import numpy as np

def check_vasprun(directory="."):
    """Find folders with complete vasprun.xml and print incomplete ones."""
    # Check if the user asked for help
    if directory == "help":
        print("Please use this function on the parent directory of the project's main folder.")
        return []
    complete_folders = []
    # Traverse all folders under "directory"
    for dirpath, _, filenames in os.walk(directory):
        if "vasprun.xml" in filenames:
            file_name_xml = os.path.join(dirpath, "vasprun.xml")

            # Check if vasprun.xml is complete
            try:
                with open(file_name_xml, "r", encoding="utf-8") as f:
                    # Check the last few lines for the closing tag
                    last_lines = f.readlines()[-10:]    # read the last 10 lines
                    for line in last_lines:
                        if "</modeling>" in line or "</vasp>" in line:
                            complete_folders.append(dirpath)
                            break
                    else:
                        print(f"vasprun.xml in {dirpath} is incomplete.")
            except IOError as e:                        # Change from Exception to IOError
                print(f"Error reading {file_name_xml}: {e}")
    return complete_folders

def specify_biolayer_lattice(directory):
    if directory == "help":
        print("Please use this function on the directory of the specific work folder and.")
        return []

    xml_path = os.path.join(directory, "vasprun.xml")
    contcar_path = os.path.join(directory, "CONTCAR")

    if os.path.isfile(xml_path) and os.path.isfile(contcar_path):
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            with open (xml_path, "r", encoding="utf-8"):
                # Extract free energy
                free_energy = float(root.findall(".//calculation/energy/i[@name='e_fr_energy']")[-1].text)

                # Extract lattice constant
                # Assuming the lattice constant is the length of the 'a' vector from the final structure
                basis_vectors = root.findall(".//calculation/structure/crystal/varray[@name='basis']")[-1]
                a_vector = basis_vectors[0].text.split()
                lattice = (float(a_vector[0])**2 + float(a_vector[1])**2 + float(a_vector[2])**2)**0.5

            coordinates = []
            with open (contcar_path, "r", encoding="utf-8") as contcar_file:
                lines = contcar_file.readlines()
                distance_bound = float(lines[4].split()[-1])
                for index, line in enumerate(lines):
                    if any(keyword in line.lower() for keyword in ["cartesian", "cart", "direct", "fractional"]):
                        coord_start_index = index + 1
                        break
                else: raise ValueError("Direct or Cartesian keyword not found in CONTCAR file.")
                total_atoms = sum([int(count) for count in lines[coord_start_index - 2].split()])
                coord_end_index = coord_start_index + total_atoms

                coordinates = lines[coord_start_index: coord_end_index]; top_layer = []; bottom_layer = []
                z_values = [float(coord.split()[-1]) for coord in coordinates]; z_average = np.mean(z_values)
                for coord in coordinates:
                    _, _, z = map(float, coord.split())
                    if z < z_average:
                        bottom_layer.append(coord)
                    else:
                        top_layer.append(coord)
                z_values_bottom = [float(coord.split()[-1]) for coord in bottom_layer]; z_average_bottom = np.mean(z_values_bottom)
                z_values_top = [float(coord.split()[-1]) for coord in top_layer]; z_average_top = np.mean(z_values_top)
                if any(keyword in line.lower() for keyword in ["cartesian", "cart"]):
                    distance = z_average_top - z_average_bottom
                if any(keyword in line.lower() for keyword in ["direct", "fractional"]):
                    distance = (z_average_top - z_average_bottom) * distance_bound
            return lattice, distance, free_energy

        except ET.ParseError as e:
            print("Error parsing vasprun.xml:", e)
            return None
    else:
        print("vasprun.xml not found in the specified directory.")
        return None

specify_biolayer_lattice("lattice_a4.600_d3.300")

def summarize_lattice_free_energy_directory(directory=".", lattice_start = None, lattice_end = None):
    result_file = "lattice_free_energy.dat"
    result_file_path = os.path.join(directory, result_file)

    if directory == "help":
        print("Please use this function on the parent directory of the project's main folder.")
        return []

    # Use check_vasprun to get folders with complete vasprun.xml
    dirs_to_walk = check_vasprun(directory)
    results = []

    for dest_dir in dirs_to_walk:
        file_name_xml = os.path.join(dest_dir, "vasprun.xml")

        if os.path.isfile(file_name_xml):
            # Extract e_fr_energy from vasprun.xml
            tree = ET.parse(file_name_xml)
            root = tree.getroot()
            e_fr_energy = float(root.findall(".//calculation/energy/i[@name='e_fr_energy']")[-1].text)
            basis_vectors = root.findall(".//calculation/structure/crystal/varray[@name='basis']")[-1]
            a_vector = basis_vectors[0].text.split()
            a_length = (float(a_vector[0])**2 + float(a_vector[1])**2 + float(a_vector[2])**2)**0.5
            lattice_constant = a_length

            # Check if lattice constant is within the specified range
            TOLERANCE = 1e-6
            within_start = lattice_start is None or lattice_constant >= lattice_start - TOLERANCE
            within_end = lattice_end is None or lattice_constant <= lattice_end + TOLERANCE
            if within_start and within_end:
                results.append((lattice_constant, e_fr_energy))

    # Sort the results by lattice_constant (the first element of the tuple)
    results.sort(key=lambda x: x[0])

    # Now write the sorted results to the file
    with open(result_file_path, "w", encoding="utf-8") as f:
        f.write("Lattice\t Free energy\n")
        for lattice_constant, e_fr_energy in results:
            f.write(f"{lattice_constant:.3f}\t{e_fr_energy}\n")

def summarize_biolayer_lattice(directory=".", lattice_start = None, lattice_end = None):
    result_file = "lattice_distance.dat"
    result_file_path = os.path.join(directory, result_file)

summarize_lattice_free_energy_directory()
